In [2]:
import sys
!{sys.executable} -m pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
    --------------------------------------- 0.0/1.5 MB 435.7 kB/s eta 0:00:04
   --- ------------------------------------ 0.1/1.5 MB 939.4 kB/s eta 0:00:02
   ----- ---------------------------------- 0.2/1.5 MB 1.3 MB/s eta 0:00:01
   -------- ------------------------------- 0.3/1.5 MB 1.5 MB/s eta 0:00:01
   ----------- ---------------------------- 0.4/1.5 MB 1.6 MB/s eta 0:00:01
   ------------- -------------------------- 0.5/1.5 MB 1.7 MB/s eta 0:00:01
   ---------------- ----------------------- 0.6/1.5 MB 1.7 MB/s eta 0:00:01
   ------------------ --------------------- 0.7/1.5 MB 1.8 MB/s eta 0:00:01
   --------------------- ------------------ 0.8/1.5 MB 1.8 MB/s eta 0:00:01
   ------------------------ --------------- 0.9/1.5 MB 1.9 MB/s eta 0:00:01
   ------------------------- -------------- 0.9/1.5 MB 1.9 MB/s eta 0:00:01
   -------------------


  Meta-Model: Riyadh
  Train: 21,912 | Val: 5,475 | Test: 5,487
  Alert rate in training: 2.4%
  Training LightGBM meta-model...
  ✓ Training complete
  ✓ Best iteration: 140

  Error Prediction Performance:
  RMSE: 36.90 W/m²
  MAE:  16.51 W/m²
  R²:   0.3250

  Anticipatory Alert Performance:
  Actual high-error windows:    289
  Predicted high-error windows: 23
  True positives (caught):      14

  Classification Report:
                  precision    recall  f1-score   support

          Normal      0.950     0.998     0.973      5198
High-Error Alert      0.609     0.048     0.090       289

        accuracy                          0.948      5487
       macro avg      0.779     0.523     0.532      5487
    weighted avg      0.932     0.948     0.927      5487


  ✓ Results saved: results/meta/riyadh_meta_results.csv
  ✓ Model saved:   models/meta/riyadh_meta_lgbm.pkl

  Meta-Model: Jeddah
  Train: 21,912 | Val: 5,475 | Test: 5,487
  Alert rate in training: 2.0%
  Training Ligh

In [5]:
# ================================================================
# 05_META_MODEL_IMPROVED.PY
# Fixing class imbalance in anticipatory forecasting
# Three techniques:
# 1. Class weights
# 2. Threshold optimization
# 3. SMOTE oversampling
# ================================================================

import sys
!{sys.executable} -m pip install imbalanced-learn lightgbm -q

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import (mean_squared_error,
                             r2_score,
                             mean_absolute_error,
                             classification_report,
                             precision_recall_curve,
                             f1_score)
from imblearn.over_sampling import SMOTE
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

os.makedirs("models/meta_improved", exist_ok=True)
os.makedirs("results/meta_improved", exist_ok=True)

# ----------------------------------------------------------------
# SETTINGS
# ----------------------------------------------------------------
ERROR_THRESHOLD  = 100   # W/m²
FORECAST_HORIZON = 3     # hours ahead

META_FEATURES = [
    "cloud_cover",
    "wind_speed",
    "humidity",
    "pressure",
    "temperature",
    "cloud_trend_3h",
    "wind_trend_3h",
    "humidity_trend_3h",
    "cloud_change_1h",
    "wind_change_1h",
    "dust_proxy",
    "wind_sin",
    "wind_cos",
    "hour",
    "month",
    "day_of_year",
    "residual_lag1",
    "residual_lag2",
    "residual_lag3",
]

# ----------------------------------------------------------------
# 1. PREPARE META LABELS (same as before)
# ----------------------------------------------------------------
def prepare_meta_labels(df, horizon=3):
    df = df.copy().sort_values("timestamp").reset_index(drop=True)
    df["meta_target"]  = df["residual"].shift(-horizon)
    df["alert_label"]  = (df["meta_target"] > ERROR_THRESHOLD).astype(int)
    df["residual_lag1"] = df["residual"].shift(1)
    df["residual_lag2"] = df["residual"].shift(2)
    df["residual_lag3"] = df["residual"].shift(3)
    df = df.dropna(subset=["meta_target",
                            "residual_lag1",
                            "residual_lag2",
                            "residual_lag3"])
    return df


# ----------------------------------------------------------------
# 2. FIND OPTIMAL THRESHOLD
# ----------------------------------------------------------------
def find_optimal_threshold(y_true, y_prob):
    """
    Instead of always using 0.5 as decision boundary,
    we find the threshold that maximizes F1 score.
    This is the balance point between precision and recall.
    """
    precisions, recalls, thresholds = precision_recall_curve(
        y_true, y_prob
    )
    f1_scores = (2 * precisions * recalls /
                 (precisions + recalls + 1e-8))
    best_idx       = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_f1        = f1_scores[best_idx]
    return best_threshold, best_f1


# ----------------------------------------------------------------
# 3. MAIN TRAINING FUNCTION
# ----------------------------------------------------------------
def train_improved_meta(city_name):
    print(f"\n{'='*55}")
    print(f"  Improved Meta-Model: {city_name}")
    print(f"{'='*55}")

    # --- Load data ---
    df = pd.read_csv(
        f"results/baseline/{city_name.lower()}_residuals.csv"
    )
    df = prepare_meta_labels(df, horizon=FORECAST_HORIZON)

    # --- Split ---
    train = df[df["split"] == "train"]
    val   = df[df["split"] == "val"]
    test  = df[df["split"] == "test"]

    X_train = train[META_FEATURES].values
    y_train = train["alert_label"].values

    X_val   = val[META_FEATURES].values
    y_val   = val["alert_label"].values

    X_test  = test[META_FEATURES].values
    y_test  = test["alert_label"].values

    # --- Class imbalance info ---
    n_normal = (y_train == 0).sum()
    n_alert  = (y_train == 1).sum()
    ratio    = n_normal / n_alert
    print(f"  Normal hours:  {n_normal:,}")
    print(f"  Alert hours:   {n_alert:,}")
    print(f"  Imbalance ratio: {ratio:.1f}:1")

    # ── FIX 1: SMOTE OVERSAMPLING ────────────────────────────────
    # SMOTE creates synthetic examples of the minority class
    # (high-error hours) so the model sees more of them
    print(f"\n  Applying SMOTE oversampling...")
    smote = SMOTE(
        sampling_strategy=0.3,  # bring alerts to 30% of normal
        random_state=42,
        k_neighbors=min(5, n_alert - 1)
    )
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    n_after = (y_train_res == 1).sum()
    print(f"  Alert hours after SMOTE: {n_after:,}")
    print(f"  New ratio: "
          f"{(y_train_res==0).sum()/(y_train_res==1).sum():.1f}:1")

    # ── FIX 2: CLASS WEIGHTS IN LIGHTGBM ────────────────────────
    # Tell LightGBM: "mistakes on alert class cost more"
    scale_pos_weight = (y_train_res == 0).sum() / \
                       (y_train_res == 1).sum()

    print(f"\n  Training improved LightGBM...")
    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=6,
        num_leaves=31,
        min_child_samples=10,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    model.fit(
        X_train_res, y_train_res,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=-1)
        ]
    )
    print(f"  ✓ Training complete")
    print(f"  ✓ Best iteration: {model.best_iteration_}")

    # ── FIX 3: OPTIMAL THRESHOLD ─────────────────────────────────
    # Get probability of alert (not just 0/1)
    val_probs  = model.predict_proba(X_val)[:, 1]
    best_thresh, best_f1 = find_optimal_threshold(y_val, val_probs)
    print(f"\n  Optimal threshold: {best_thresh:.3f} "
          f"(default was 0.500)")
    print(f"  Best F1 on val:    {best_f1:.3f}")

    # ── EVALUATE ON TEST SET ─────────────────────────────────────
    test_probs    = model.predict_proba(X_test)[:, 1]
    test_preds    = (test_probs >= best_thresh).astype(int)

    # Default threshold results (for comparison)
    test_preds_default = (test_probs >= 0.5).astype(int)

    print(f"\n  {'─'*45}")
    print(f"  RESULTS COMPARISON")
    print(f"  {'─'*45}")

    # Before fix
    f1_default = f1_score(y_test, test_preds_default)
    recall_default = (
        ((test_preds_default == 1) & (y_test == 1)).sum() /
        (y_test == 1).sum()
    )

    # After fix
    f1_improved = f1_score(y_test, test_preds)
    recall_improved = (
        ((test_preds == 1) & (y_test == 1)).sum() /
        (y_test == 1).sum()
    )

    actual_alerts    = (y_test == 1).sum()
    caught_alerts    = ((test_preds == 1) & (y_test == 1)).sum()
    predicted_alerts = (test_preds == 1).sum()
    precision_val    = (caught_alerts / predicted_alerts
                        if predicted_alerts > 0 else 0)

    print(f"\n  Actual high-error windows: {actual_alerts:,}")
    print(f"\n  DEFAULT threshold (0.5):")
    print(f"    Recall:  {recall_default:.1%}  "
          f"← how many failures caught")
    print(f"    F1:      {f1_default:.3f}")
    print(f"\n  IMPROVED threshold ({best_thresh:.3f}):")
    print(f"    Recall:  {recall_improved:.1%}  "
          f"← how many failures caught")
    print(f"    F1:      {f1_improved:.3f}")
    print(f"    Caught:  {caught_alerts} / {actual_alerts}")
    print(f"    Precision: {precision_val:.1%}")

    print(f"\n  Full Classification Report:")
    print(classification_report(
        y_test, test_preds,
        target_names=["Normal", "High-Error Alert"],
        digits=3
    ))

    # ── SAVE RESULTS ─────────────────────────────────────────────
    test_copy = test.copy()
    test_copy["alert_prob"]      = test_probs
    test_copy["predicted_alert"] = test_preds
    test_copy["actual_alert"]    = y_test

    out_path = (f"results/meta_improved/"
                f"{city_name.lower()}_improved_results.csv")
    test_copy.to_csv(out_path, index=False)

    model_path = (f"models/meta_improved/"
                  f"{city_name.lower()}_improved_lgbm.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(model, f)

    print(f"\n  ✓ Saved: {out_path}")

    return {
        "city":            city_name,
        "actual_alerts":   int(actual_alerts),
        "caught_alerts":   int(caught_alerts),
        "recall_default":  round(recall_default, 3),
        "recall_improved": round(recall_improved, 3),
        "f1_default":      round(f1_default, 3),
        "f1_improved":     round(f1_improved, 3),
        "precision":       round(float(precision_val), 3),
        "threshold":       round(float(best_thresh), 3),
    }


# ----------------------------------------------------------------
# 4. RUN ALL CITIES
# ----------------------------------------------------------------
all_results = []

for city in ["Riyadh", "Jeddah", "Abha", "Dammam", "Qassim"]:
    result = train_improved_meta(city)
    all_results.append(result)


# ----------------------------------------------------------------
# 5. FINAL COMPARISON TABLE
# ----------------------------------------------------------------
results_df = pd.DataFrame(all_results)

print(f"\n{'='*55}")
print("  IMPROVEMENT SUMMARY")
print(f"{'='*55}")
print(f"\n  {'City':<10} {'Recall':>12} {'Recall':>12} "
      f"{'F1':>8} {'F1':>8}")
print(f"  {'':10} {'Before':>12} {'After':>12} "
      f"{'Before':>8} {'After':>8}")
print(f"  {'─'*52}")

for _, r in results_df.iterrows():
    print(f"  {r['city']:<10} "
          f"{r['recall_default']:>11.1%} "
          f"{r['recall_improved']:>11.1%} "
          f"{r['f1_default']:>8.3f} "
          f"{r['f1_improved']:>8.3f}")

results_df.to_csv(
    "results/meta_improved/improvement_summary.csv",
    index=False
)

print(f"\n  ✓ ALL CITIES IMPROVED")
print(f"  ✓ Ready for SHAP Analysis (Step 6)")


  Improved Meta-Model: Riyadh
  Normal hours:  21,396
  Alert hours:   516
  Imbalance ratio: 41.5:1

  Applying SMOTE oversampling...
  Alert hours after SMOTE: 6,418
  New ratio: 3.3:1

  Training improved LightGBM...
  ✓ Training complete
  ✓ Best iteration: 434

  Optimal threshold: 0.240 (default was 0.500)
  Best F1 on val:    0.416

  ─────────────────────────────────────────────
  RESULTS COMPARISON
  ─────────────────────────────────────────────

  Actual high-error windows: 289

  DEFAULT threshold (0.5):
    Recall:  42.6%  ← how many failures caught
    F1:      0.421

  IMPROVED threshold (0.240):
    Recall:  67.1%  ← how many failures caught
    F1:      0.460
    Caught:  194 / 289
    Precision: 35.0%

  Full Classification Report:
                  precision    recall  f1-score   support

          Normal      0.981     0.931     0.955      5198
High-Error Alert      0.350     0.671     0.460       289

        accuracy                          0.917      5487
      